# NLP Assignment 2

**How it works**

1. **Upload** a PDF.
2. The text is extracted, cleaned of junk characters, and split into sentences (with page numbers).
3. **Retrieval:** a sentence-embedding model (`all-MiniLM-L6-v2`) finds the sentences most similar to your question.
4. **Reading:** an extractive QA model (`deepset/roberta-base-squad2`, trained to detect *unanswerable* questions) pulls the answer span out of those sentences (plus their neighbours). If no short span fits but a clearly matching sentence exists, that sentence is returned.
5. If the PDF has no good evidence, the tool replies: **"I don't have proper context to answer that question."**

**Run all cells from top to bottom** (Runtime -> Run all in Colab). The last cell asks you to upload a PDF, then asks for your questions.

## 1. Install dependencies

In [1]:
!pip install -q pypdf transformers sentence-transformers torch ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 72.1 MB/s eta 0:00:00


## 2. Imports and models


In [2]:
import re
import io
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

print("Loading retrieval model...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Loading question-answering model...")
QA_MODEL_NAME = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
qa_model.eval()

print("Models ready.")

Loading retrieval model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  496MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Models ready.


## 3. Core logic

In [5]:
MAX_SENT_WORDS = 60        # very long sentences are split into pieces of at most this many words
TOP_K = 5                  # how many candidate sentences are examined
CONTEXT_WINDOW = 1         # neighbouring sentences added on each side for the QA model
MIN_RETRIEVAL_SIM = 0.25   # best sentence below this similarity -> nothing related in the PDF
MIN_ANSWER_SCORE = 0.50    # QA-model confidence needed to give a short extracted answer
MIN_SENTENCE_SIM = 0.40    # similarity needed to return a whole sentence as the answer (fallback)

NO_CONTEXT_MSG = "I don't have proper context to answer that question."

# Words that carry no topic information. Used to find the "key words" of a question.
STOP_WORDS = set("""
a an the of in on at to for from by with and or but as is are was were be been being am do does did done
have has had it its this that these those there their they them he she his her we our you your i me my
what which who whom whose when where why how can could should would will shall may might must tell
please give explain describe define about than then so if not no yes also very more most any some such
into over under between during after before up down out off again further once here both each few other
own same too just only mean means meant
""".split())
GENERIC_WORDS = set("""
purpose purposes important importance crucial vital essential significant significance major main primary
key role roles used use uses using useful name names called known kind kinds type types reason reasons
thing things way ways much many one
""".split())
IGNORED_WORDS = STOP_WORDS | GENERIC_WORDS


# ---------- PDF reading and cleaning ----------
def extract_pages(pdf_bytes):
    """Return a list of (page_number, text) from PDF bytes."""
    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            pages.append((i, text))
    return pages


JUNK_RUN = re.compile(r"[\$%&\(\)_\*#@\^~=\+\|<>\{\}\[\]\\/`]{2,}")


def clean_text(text):
    """Remove decorative junk such as '$$$$%%%%&&&&' or '(_ ... _)' that some PDFs contain."""
    text = JUNK_RUN.sub(" ", text)
    text = re.sub(r"(?<!\w)[_\$%&\*#@\^~=\|<>\\`]+(?!\w)", " ", text)   # lone leftover symbols
    text = re.sub(r"(^|\s)[.,;:]+(?=\s|$)", " ", text)                  # orphan punctuation
    return re.sub(r"\s+", " ", text).strip()


SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[\"'“‘(\[]?[A-Z0-9])")


def split_sentences(text):
    out = []
    for part in SENT_SPLIT.split(text):
        words = part.split()
        for i in range(0, len(words), MAX_SENT_WORDS):
            piece = " ".join(words[i:i + MAX_SENT_WORDS])
            if piece:
                out.append(piece)
    return out


def build_sentences(pages):
    sents = []
    for page_no, text in pages:
        for s in split_sentences(text):
            if not re.search(r"[A-Za-z]", s):
                continue
            if sents and sents[-1]["page"] == page_no and len(s.split()) < 4:
                sents[-1]["text"] += " " + s          # glue tiny fragments to the previous sentence
            else:
                sents.append({"page": page_no, "text": s})
    return sents


# ---------- Key-word helpers (used to avoid answering from unrelated text) ----------
def stem(word):
    word = word.lower()
    for suf in ("ing", "ed", "es", "s"):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[: -len(suf)]
    return word


def question_terms(question):
    """Key words of a question, e.g. 'What is the national bird of Bangladesh?' -> {national, bird, bangladesh}"""
    return {
        stem(t) for t in re.findall(r"[A-Za-z0-9]+", question.lower())
        if len(t) >= 3 and t not in IGNORED_WORDS
    }


def text_stems(text):
    return {stem(t) for t in re.findall(r"[A-Za-z0-9]+", text.lower()) if len(t) >= 3}


def term_in(term, stems):
    if term in stems:
        return True
    if len(term) >= 4 and any(d.startswith(term) for d in stems):
        return True
    return any(len(d) >= 5 and term.startswith(d) for d in stems)


def keywords_supported(question, doc_stems):
    """
    True if the key words of the question actually appear in the PDF.
    'What is the national bird of Bangladesh?' has the key word 'bird'; if the PDF never
    mentions birds, the question is rejected even if the model would guess 'tiger'.
    Long questions (4+ key words) are allowed to miss one word.
    """
    terms = question_terms(question)
    missing = [t for t in terms if not term_in(t, doc_stems)]
    allowed_missing = 1 if len(terms) >= 4 else 0
    return len(missing) <= allowed_missing


# ---------- Extractive QA with the SQuAD2 model ----------
def extract_answer(question, context, max_answer_tokens=48, n_best=20):
    """
    Returns (answer_text, confidence, char_start, char_end).
    Confidence = probability that a real answer exists in the context rather than
    the model's "no answer" option (the [CLS] position).
    """
    enc = qa_tokenizer(
        question, context,
        return_tensors="pt", truncation="only_second",
        max_length=512, return_offsets_mapping=True,
    )
    offsets = enc.pop("offset_mapping")[0].tolist()
    seq_ids = enc.sequence_ids(0)
    with torch.no_grad():
        out = qa_model(**enc)
    start_logits = out.start_logits[0].numpy()
    end_logits = out.end_logits[0].numpy()

    null_score = float(start_logits[0] + end_logits[0])      # "no answer"
    ctx_positions = [i for i, s in enumerate(seq_ids) if s == 1]
    if not ctx_positions:
        return "", 0.0, 0, 0

    start_cands = sorted(ctx_positions, key=lambda i: -start_logits[i])[:n_best]
    end_cands = sorted(ctx_positions, key=lambda i: -end_logits[i])[:n_best]

    best_score, best_span = -1e9, None
    for s in start_cands:
        for e in end_cands:
            if e < s or (e - s + 1) > max_answer_tokens:
                continue
            score = float(start_logits[s] + end_logits[e])
            if score > best_score:
                best_score, best_span = score, (s, e)

    if best_span is None:
        return "", 0.0, 0, 0

    confidence = float(1.0 / (1.0 + np.exp(null_score - best_score)))
    s, e = best_span
    char_start, char_end = offsets[s][0], offsets[e][1]
    return context[char_start:char_end].strip(), confidence, char_start, char_end


# ---------- The question-answering engine ----------
class PDFQA:
    def __init__(self, pdf_bytes):
        pages = [(n, clean_text(t)) for n, t in extract_pages(pdf_bytes)]
        pages = [(n, t) for n, t in pages if t]
        if not pages:
            raise ValueError(
                "No readable text found in this PDF. It may be a scanned image "
                "(this tool needs a text-based PDF, or run OCR on it first)."
            )
        self.sents = build_sentences(pages)
        if not self.sents:
            raise ValueError("Could not find any sentences in this PDF.")
        self.n_pages = len(pages)
        self.doc_stems = text_stems(" ".join(s["text"] for s in self.sents))
        self.embeddings = embedder.encode(
            [s["text"] for s in self.sents],
            convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
        )

    def _window(self, idx):
        """The sentence at idx plus its neighbours on the same page."""
        page = self.sents[idx]["page"]
        lo = hi = idx
        for _ in range(CONTEXT_WINDOW):
            if lo - 1 >= 0 and self.sents[lo - 1]["page"] == page:
                lo -= 1
            if hi + 1 < len(self.sents) and self.sents[hi + 1]["page"] == page:
                hi += 1
        return " ".join(s["text"] for s in self.sents[lo:hi + 1])

    def answer(self, question):
        q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)[0]
        sims = self.embeddings @ q_emb
        top_idx = [int(i) for i in np.argsort(-sims)[:TOP_K]]

        # Nothing in the document is even related to the question
        if sims[top_idx[0]] < MIN_RETRIEVAL_SIM:
            return {"found": False}

        # The question asks about something (e.g. "bird") that the PDF never mentions
        if not keywords_supported(question, self.doc_stems):
            return {"found": False}

        # Tier 1: short extracted answer from the QA model
        best = None
        for idx in top_idx:
            if sims[idx] < MIN_RETRIEVAL_SIM:
                continue
            context = self._window(idx)
            ans, conf, c_start, c_end = extract_answer(question, context)
            if not ans:                       # model says "no answer here"
                continue
            combined = conf * (0.5 + 0.5 * float(sims[idx]))
            if best is None or combined > best["combined"]:
                best = {
                    "mode": "span", "answer": ans, "qa_score": conf, "combined": combined,
                    "page": self.sents[idx]["page"], "context": context,
                    "start": c_start, "end": c_end,
                }
        if best is not None and best["qa_score"] >= MIN_ANSWER_SCORE:
            best["found"] = True
            return best

        terms = question_terms(question)
        for idx in top_idx:
            if sims[idx] < MIN_SENTENCE_SIM:
                continue
            text = self.sents[idx]["text"]
            if terms and not any(term_in(t, text_stems(text)) for t in terms):
                continue
            nxt = idx + 1
            if nxt < len(self.sents) and self.sents[nxt]["page"] == self.sents[idx]["page"]:
                text = text + " " + self.sents[nxt]["text"]
            return {
                "found": True, "mode": "sentence", "answer": text,
                "page": self.sents[idx]["page"], "sim": float(sims[idx]),
            }
        return {"found": False}


def format_result(res):
    if not res["found"]:
        return NO_CONTEXT_MSG
    if res.get("mode") == "sentence":
        return f"Answer: {res['answer']}\nPage: {res['page']}"
    ctx = res["context"]
    s, e = res["start"], res["end"]
    snippet = ctx[max(0, s - 120):min(len(ctx), e + 120)]
    return (
        f"Answer: {res['answer']}\n"
        f"Confidence: {res['qa_score']:.0%}   |   Page: {res['page']}\n"
        f"Evidence: ...{snippet}..."
    )


print("Core functions defined.")

Core functions defined.


## 4. Run the tool

After processing, you will be asked for your question. Type `exit` (Colab) to stop.

In [6]:
def run_in_colab():
    from google.colab import files
    print("Please upload your PDF file...")
    uploaded = files.upload()
    if not uploaded:
        print("No file uploaded. Run this cell again.")
        return
    name = next(iter(uploaded))
    if not name.lower().endswith(".pdf"):
        print(f"'{name}' is not a PDF. Run this cell again and upload a .pdf file.")
        return
    print(f"\nUpload complete: {name}. Reading the PDF...")
    try:
        qa = PDFQA(uploaded[name])
    except Exception as ex:
        print("Error:", ex)
        return
    print(f"Ready! {qa.n_pages} page(s) with text, {len(qa.sents)} sentences indexed.\n")

    while True:
        question = input("Your question (or type 'exit'): ").strip()
        if question.lower() in {"exit", "quit", "q"}:
            print("Goodbye!")
            break
        if not question:
            continue
        print()
        print(format_result(qa.answer(question)))
        print("-" * 70)


def run_in_jupyter():
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    state = {"qa": None}
    upload = widgets.FileUpload(accept=".pdf", multiple=False, description="Upload PDF")
    status = widgets.Output()
    qbox = widgets.Text(placeholder="Type your question here...", description="Question:",
                        layout=widgets.Layout(width="80%"))
    ask_btn = widgets.Button(description="Ask", button_style="primary")
    result = widgets.Output()
    qa_area = widgets.VBox([qbox, ask_btn, result], layout=widgets.Layout(display="none"))

    def get_bytes(value):
        # ipywidgets 7 returns a dict, ipywidgets 8 returns a tuple of dicts
        item = list(value.values())[0] if isinstance(value, dict) else value[0]
        return item["content"] if isinstance(item["content"], (bytes, bytearray)) else bytes(item["content"])

    def on_upload(change):
        with status:
            clear_output()
            print("Upload complete. Reading the PDF...")
            try:
                state["qa"] = PDFQA(get_bytes(upload.value))
            except Exception as ex:
                print("Error:", ex)
                qa_area.layout.display = "none"
                return
            qa = state["qa"]
            print(f"Ready! {qa.n_pages} page(s) with text, {len(qa.sents)} sentences indexed.")
            print("Now type your question below.")
        qa_area.layout.display = "flex"

    def on_ask(_):
        with result:
            clear_output()
            q = qbox.value.strip()
            if not q:
                print("Please type a question.")
                return
            print("Thinking...")
            out = format_result(state["qa"].answer(q))
            clear_output()
            print("Q:", q)
            print()
            print(out)

    upload.observe(on_upload, names="value")
    ask_btn.on_click(on_ask)
    qbox.on_submit(on_ask)
    display(widgets.VBox([upload, status, qa_area]))


try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    run_in_colab()
else:
    run_in_jupyter()

Please upload your PDF file...


Saving Test PDF (Hard).pdf to Test PDF (Hard) (1).pdf

Upload complete: Test PDF (Hard) (1).pdf. Reading the PDF...
Ready! 2 page(s) with text, 23 sentences indexed.

Your question (or type 'exit'): What is the national bird of Bangladesh?

I don't have proper context to answer that question.
----------------------------------------------------------------------
Your question (or type 'exit'): For what purpose Padma river is crucial for?  I don't have proper context to answer that question.

I don't have proper context to answer that question.
----------------------------------------------------------------------
Your question (or type 'exit'): For what purpose Padma river is crucial for?

Answer: transportation, agriculture, fishing, and daily life
Confidence: 100%   |   Page: 1
Evidence: ... of Bangladesh is the Padma River, which is often referred to as the lifeline of the country. This river is crucial for transportation, agriculture, fishing, and daily life....
-------------------